In [ ]:
# ============================================================
#  Federated Learning — FedOpt (FedAdam)
#  Dataset   : PlantVillage Diseases dataset
#  Rounds    : 50  |  Clients : 5  |  Backbone : ResNet18
# ============================================================

# ── 0. Imports ───────────────────────────────────────────────
import os, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import torchvision.transforms as transforms
from torchvision import datasets, models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
torch.manual_seed(42);  np.random.seed(42);  random.seed(42)


# ── 1. Config ────────────────────────────────────────────────
class Config:
    # ── Correct path (class folders sit directly inside) ──
    DATA_ROOT       = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"

    # Federated (paper §4.1-4.4)
    NUM_CLIENTS     = 5
    DIRICHLET_ALPHA = 0.5
    NUM_ROUNDS      = 50
    LOCAL_EPOCHS    = 5
    BATCH_SIZE      = 32
    LR              = 0.0001          # client SGD lr
    MOMENTUM        = 0.9

    # FedOpt server-side Adam
    SERVER_LR       = 0.001          # server Adam lr  (η)
    SERVER_BETA1    = 0.9
    SERVER_BETA2    = 0.99
    SERVER_EPS      = 1e-3           # τ  (kept > 0 for stability)

    DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    IMG_SIZE        = 224
    SAVE_DIR        = "/kaggle/working"

cfg = Config()
print(f"Device  : {cfg.DEVICE}")
print(f"Classes : {sorted(os.listdir(cfg.DATA_ROOT))}")


# ── 2. Transforms ────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
test_tf = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


# ── 3. Dataset ───────────────────────────────────────────────
class PathDataset(Dataset):
    """ImageFolder-compatible dataset that always applies its own transform."""
    def __init__(self, samples, class_to_idx, transform):
        self.samples      = samples        # list of (path, label_int)
        self.class_to_idx = class_to_idx
        self.transform    = transform
        self.targets      = [s[1] for s in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


# Build master sample list
_ref = datasets.ImageFolder(cfg.DATA_ROOT)
CLASS_NAMES  = _ref.classes
NUM_CLASSES  = len(CLASS_NAMES)
ALL_SAMPLES  = _ref.samples           # list of (path, int)
ALL_TARGETS  = np.array(_ref.targets)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Total   : {len(ALL_SAMPLES)} images")

# 80/20 stratified split
all_idx = np.arange(len(ALL_SAMPLES))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42, stratify=ALL_TARGETS)

train_samples = [ALL_SAMPLES[i] for i in train_idx]
test_samples  = [ALL_SAMPLES[i] for i in test_idx]

train_ds = PathDataset(train_samples, _ref.class_to_idx, train_tf)
test_ds  = PathDataset(test_samples,  _ref.class_to_idx, test_tf)
print(f"Train : {len(train_ds)} | Test : {len(test_ds)}")


# ── 4. Dirichlet non-IID partition ──────────────────────────
def dirichlet_partition(dataset, num_clients, alpha, num_classes):
    targets    = np.array(dataset.targets)
    class_idx  = defaultdict(list)
    for i, t in enumerate(targets):
        class_idx[t].append(i)

    client_idx = defaultdict(list)
    for cls in range(num_classes):
        idxs = np.array(class_idx[cls]);  np.random.shuffle(idxs)
        props = np.random.dirichlet(np.repeat(alpha, num_clients))
        props = np.maximum(props, 1e-6);  props /= props.sum()
        splits = (props * len(idxs)).astype(int)
        splits[-1] = max(0, len(idxs) - splits[:-1].sum())
        start = 0
        for cid, cnt in enumerate(splits):
            client_idx[cid].extend(idxs[start:start+cnt].tolist())
            start += cnt
    return dict(client_idx)

client_map = dirichlet_partition(train_ds, cfg.NUM_CLIENTS,
                                 cfg.DIRICHLET_ALPHA, NUM_CLASSES)

print("\n── Client distribution ──")
for cid, idxs in client_map.items():
    lbls = [train_ds.targets[i] for i in idxs]
    dist = {CLASS_NAMES[c]: lbls.count(c) for c in range(NUM_CLASSES)}
    print(f"  Client {cid}: {len(idxs)} samples | {dist}")


# ── 5. DataLoaders ───────────────────────────────────────────
def make_client_loader(cid):
    return DataLoader(Subset(train_ds, client_map[cid]),
                      batch_size=cfg.BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)

client_loaders = {cid: make_client_loader(cid) for cid in range(cfg.NUM_CLIENTS)}
test_loader    = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)


# ── 6. Model ─────────────────────────────────────────────────
def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(cfg.DEVICE)


# ── 7. Evaluation ────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, truths = [], []
    for imgs, labels in loader:
        imgs = imgs.to(cfg.DEVICE)
        out  = model(imgs)
        preds.extend(out.argmax(1).cpu().numpy())
        truths.extend(labels.numpy())
    acc  = accuracy_score(truths, preds)
    prec = precision_score(truths, preds, average='weighted', zero_division=0)
    rec  = recall_score(truths, preds, average='weighted', zero_division=0)
    f1   = f1_score(truths, preds, average='weighted', zero_division=0)
    return acc, prec, rec, f1, truths, preds


# ── 8. Local training ─────────────────────────────────────────
def local_train(model, loader, epochs):
    model.train()
    opt  = optim.SGD(model.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
            opt.zero_grad()
            crit(model(imgs), labels).backward()
            opt.step()
    return {k: v.clone() for k, v in model.state_dict().items()}


# ══════════════════════════════════════════════════════════════
#  FedOpt  — Server-side Adam optimiser on pseudo-gradient
# ══════════════════════════════════════════════════════════════
#
#  Algorithm (FedAdam variant of FedOpt, Reddi et al. 2021):
#
#  Each round t:
#    1. Broadcast  w_t  to all clients
#    2. Each client k runs SGD for E epochs → w_t^k
#    3. Compute pseudo-gradient (delta) per client:
#           Δ_k = w_t - w_t^k            (descent direction)
#    4. Aggregate deltas (weighted avg):
#           Δ = Σ_k (n_k/N) · Δ_k
#    5. Server Adam update on the GLOBAL model:
#           m_t   = β1·m_{t-1}  + (1-β1)·Δ
#           v_t   = β2·v_{t-1}  + (1-β2)·Δ²
#           w_{t+1} = w_t  -  η · m_t / (√v_t + τ)
#
#  Key difference vs FedAvg:
#    FedAvg  : w_{t+1} = Σ_k (n_k/N)·w_t^k       (weight averaging)
#    FedOpt  : w_{t+1} = w_t - Adam(Δ)             (server optimiser)
# ══════════════════════════════════════════════════════════════

class ServerAdam:
    """
    Maintains Adam first & second moment vectors for every
    parameter key in the model's state_dict.
    All tensors live on CPU to keep memory usage low;
    we cast to device only when needed for arithmetic.
    """
    def __init__(self, model_sd, lr, beta1, beta2, eps):
        self.lr    = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps   = eps
        self.t     = 0

        # Initialise moments to zero (CPU)
        self.m = {k: torch.zeros_like(v, dtype=torch.float32)
                  for k, v in model_sd.items()
                  if v.dtype in (torch.float32, torch.float16, torch.bfloat16)}
        self.v = {k: torch.zeros_like(v, dtype=torch.float32)
                  for k, v in model_sd.items()
                  if v.dtype in (torch.float32, torch.float16, torch.bfloat16)}

    def step(self, global_sd, delta_sd):
        """
        Apply one Adam step.
        global_sd : current global state dict (CPU tensors)
        delta_sd  : aggregated pseudo-gradient  (CPU tensors)
        Returns   : updated state dict (CPU tensors)
        """
        self.t += 1
        new_sd = copy.deepcopy(global_sd)

        for key in global_sd:
            if key not in self.m:          # int/bool buffers (e.g. num_batches_tracked)
                continue
            if key not in delta_sd:
                continue

            g = delta_sd[key].float()     # pseudo-gradient

            # Bias-corrected moments
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * g
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * g * g

            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)

            new_sd[key] = (global_sd[key].float()
                           - self.lr * m_hat / (v_hat.sqrt() + self.eps))

            # Cast back to original dtype
            new_sd[key] = new_sd[key].to(global_sd[key].dtype)

        return new_sd


def aggregate_deltas(global_sd, client_sds, client_sizes):
    """
    Weighted average of pseudo-gradients Δ_k = w_global - w_k^local.
    Returns delta_sd on CPU.
    """
    total    = sum(client_sizes)
    delta_sd = {}
    for key in global_sd:
        if global_sd[key].dtype not in (torch.float32, torch.float16, torch.bfloat16):
            continue
        delta = torch.zeros_like(global_sd[key], dtype=torch.float32)
        for sd, sz in zip(client_sds, client_sizes):
            # Δ_k = w_global - w_k  (on CPU)
            delta += (global_sd[key].float() - sd[key].cpu().float()) * (sz / total)
        delta_sd[key] = delta
    return delta_sd


def run_fedopt():
    print("\n" + "="*60)
    print("  Running FedOpt (FedAdam)  —  50 rounds")
    print(f"  Server Adam: lr={cfg.SERVER_LR}  β1={cfg.SERVER_BETA1}"
          f"  β2={cfg.SERVER_BETA2}  ε={cfg.SERVER_EPS}")
    print("="*60)

    global_model = build_model()

    # Move global sd to CPU for moment storage
    global_sd = {k: v.cpu() for k, v in global_model.state_dict().items()}

    server_opt = ServerAdam(
        global_sd,
        lr    = cfg.SERVER_LR,
        beta1 = cfg.SERVER_BETA1,
        beta2 = cfg.SERVER_BETA2,
        eps   = cfg.SERVER_EPS,
    )

    history = {"round":[], "accuracy":[], "precision":[], "recall":[], "f1":[]}
    best_acc   = 0.0
    best_sd    = None

    for rnd in range(1, cfg.NUM_ROUNDS + 1):

        # ── Broadcast & local train ──────────────────────────
        client_sds, client_sizes = [], []
        for cid in range(cfg.NUM_CLIENTS):
            local_model = build_model()
            # Load current global weights onto device
            local_model.load_state_dict(
                {k: v.to(cfg.DEVICE) for k, v in global_sd.items()})
            sd = local_train(local_model, client_loaders[cid], cfg.LOCAL_EPOCHS)
            client_sds.append({k: v.cpu() for k, v in sd.items()})
            client_sizes.append(len(client_map[cid]))

        # ── Aggregate pseudo-gradients ────────────────────────
        delta_sd = aggregate_deltas(global_sd, client_sds, client_sizes)

        # ── Server Adam step ──────────────────────────────────
        global_sd = server_opt.step(global_sd, delta_sd)

        # ── Evaluate ─────────────────────────────────────────
        global_model.load_state_dict(
            {k: v.to(cfg.DEVICE) for k, v in global_sd.items()})
        acc, prec, rec, f1, _, _ = evaluate(global_model, test_loader)

        history["round"].append(rnd)
        history["accuracy"].append(acc)
        history["precision"].append(prec)
        history["recall"].append(rec)
        history["f1"].append(f1)

        # Track best
        if acc > best_acc:
            best_acc = acc
            best_sd  = copy.deepcopy(global_sd)

        if rnd % 5 == 0 or rnd == 1:
            print(f"  Round {rnd:3d} | Acc={acc:.4f}  Prec={prec:.4f}  "
                  f"Rec={rec:.4f}  F1={f1:.4f}  [best={best_acc:.4f}]")

    # Restore best weights
    print(f"\n  Restoring best model (Acc={best_acc:.4f})")
    global_model.load_state_dict(
        {k: v.to(cfg.DEVICE) for k, v in best_sd.items()})

    return global_model, history


fedopt_model, fedopt_hist = run_fedopt()


# ── 9. Final evaluation ──────────────────────────────────────
print("\n" + "="*60)
print("  FEDOPT — FINAL TEST RESULTS")
print("="*60)
acc, prec, rec, f1, y_true, y_pred = evaluate(fedopt_model, test_loader)
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


# ── 10. Plots ─────────────────────────────────────────────────
# Training curves
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("FedOpt (FedAdam) — Training Curves | tomatato Disease", fontsize=14)
colors = {"accuracy":"#1f77b4","precision":"#ff7f0e","recall":"#2ca02c","f1":"#d62728"}
for ax, metric, label in zip(
        axes.flatten(),
        ["accuracy","precision","recall","f1"],
        ["Accuracy","Precision","Recall","F1-Score"]):
    ax.plot(fedopt_hist["round"], fedopt_hist[metric],
            marker='o', color=colors[metric], linewidth=2)
    ax.set_title(label)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(label)
    ax.set_xlim(left=1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, "fedopt_training_curves.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fedopt_training_curves.png")

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title("FedOpt — Confusion Matrix")
ax.set_xlabel("Predicted");  ax.set_ylabel("True")
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, "fedopt_confusion_matrix.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fedopt_confusion_matrix.png")

# Round-by-round metrics table
hist_df = pd.DataFrame(fedopt_hist)
hist_df.to_csv(os.path.join(cfg.SAVE_DIR, "fedopt_history.csv"), index=False)
print("\nPer-round metrics:")
print(hist_df.to_string(index=False))

# Save model
torch.save(fedopt_model.state_dict(),
           os.path.join(cfg.SAVE_DIR, "fedopt_plantvillage.pth"))
print("\nModel saved: fedopt_potato.pth")
print("\nFedOpt Done ✓")